# Matched-transformer control — byte vs subword at EQUAL transformer params

The reviewer rebuttal: *is byte's advantage just more parameters?* This run removes the confound. At each size both variants are built from the **same mT5-{size} transformer config** (random-init), so the transformer parameter count is **byte-for-byte identical** — only `vocab_size` (384 bytes vs 250k subwords) and the tokenizer differ. Verified locally: transformer = **19.41M** (small) / **85.74M** (base) for *both*; byte's table is 0.2M vs subword's 128M, so byte even has **far fewer total params**.

Same SONAR distillation as the main runs (reuses the cached targets) and same eval battery. **From scratch / random-init** — no pretrained byte/subword transformer is shared-compatible — so this is a clean isolating ablation: fair on *both* axes (matched transformer, matched zero-pretraining). **Absolute scores are lower than the pretrained main runs by design; the signal is the byte−subword delta at matched transformer.** Smoke cell first.

### 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. SONAR teacher + Drive persist
Point at the **same** `byteembed_lowres` folder as the main runs so this reuses the **cached SONAR targets** (`teachertargets_sonar_9langs_42000.npy`) — no teacher reload, no re-embedding. Skip Drive to run on ephemeral disk (it will then load SONAR + embed once).

In [ ]:
from transformers import AutoTokenizer
from transformers.models.m2m_100.modeling_m2m_100 import M2M100Encoder
_ = AutoTokenizer.from_pretrained('cointegrated/SONAR_200_text_encoder')   # SONAR HF port reachable
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the main runs -> shared cache
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)

### 4. Smoke test (~5 min) — validate the whole pipeline
3 langs (am/rw/en), one tiny byte student, 80 steps. Confirms build-from-config → distill → eval works and prints the param split (transformer matched).

In [ ]:
from byte_embed.run_matched import run
_ = run(smoke=True, out='results/matched_smoke.json')

### 5. Full matched-transformer run
Trains **byte-small, subword-small, byte-base, subword-base** at matched transformer (mT5-small/base dims, random-init), 50k SONAR-distillation steps each, mean pooling (same for both → fair). Resumable. The param split prints per model so you can confirm the transformer is equal; the summary prints the byte−subword delta at matched transformer.

In [ ]:
from byte_embed.run_matched import run
_ = run(
    out='results/matched.json',
    sizes=('small', 'base'),   # add 'large' for mt5-large dims (~310M transformer) if you want it
    steps=50000,
    pooling='mean',
)

### 6. Results — matched-transformer table + byte−subword deltas

In [ ]:
import json
from byte_embed.run_matched import _summary
_summary(json.load(open('results/matched.json')))

### 7. Download results

In [ ]:
from google.colab import files
files.download('results/matched.json')